# `gold_analytics.ipynb` - Capa Gold (modelado dimensional y metricas)

Calcula, con DuckDB consultando directamente los ficheros Parquet de
Silver, las tres metricas analiticas exigidas por el enunciado
(requisito funcional 3):

1. **Resumen diario agregado por ubicacion**: temperatura media, minima,
   maxima, lluvia total acumulada y velocidad media del viento.
2. **Media movil continua a 24 horas** de temperatura (metrica horaria).
3. **Z-Score horario por ciudad frente a su historico cargado**, para
   detectar desviaciones termicas.

> **Decision de diseno (¿que es el "historico cargado"?)**: Open-Meteo
> `/v1/forecast` sirve prediccion, no un registro historico externo. En
> lugar de inventar un dataset de normales climatologicas, el "historico
> cargado" se define aqui como **toda la serie ya acumulada en Silver
> para esa ciudad** (creciente en cada ejecucion real del pipeline): la
> media y la desviacion tipica de temperatura se calculan sobre esa
> serie completa, y el Z-Score de cada hora se calcula frente a esas
> media/desviacion. Es una definicion honesta con datos 100% reales (los
> que el propio pipeline ha ido acumulando), en vez de una referencia
> externa que no se puede descargar de forma verificada en este entorno
> (ver README, seccion "Notas importantes").

> **Decision de diseno (modelado dimensional)**: las tres tablas Gold se
> escriben en formato Parquet, particionadas por ciudad, y se
> sobrescriben por completo en cada ejecucion (se recalculan desde toda
> la Silver disponible), lo que las hace idempotentes por construccion:
> repetir la ejecucion sin cambios en Silver produce exactamente el mismo
> resultado.

> Depende de: `config.ipynb`, `logging_config.ipynb`. Se ejecuta despues
> de `silver_clean.ipynb` (lee directamente de `data/silver/`, no
> encadena su ejecucion con `%run` para poder recalcularse de forma
> independiente sobre lo que ya hay en disco).

In [1]:
import os
if os.getcwd().endswith("climate_engine"):
    os.chdir("..")
%run climate_engine/config.ipynb
%run climate_engine/logging_config.ipynb

Fichero de configuracion: /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco/config.yaml
Raiz del proyecto:        /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco
Ubicaciones cargadas (5): ['Madrid', 'Barcelona', 'Valencia', 'Sevilla', 'Bilbao']
Variables horarias:       ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'surface_pressure']
Ventana de prediccion:    14 dias, zona horaria UTC
Politica de reintentos:   3 reintentos, backoff x2, codigos [429, 500, 502, 503, 504]
Rango temperatura valido: (-50.0, 60.0)
Rango humedad valido:     (0.0, 100.0)
Ruta bronze:  /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco/data/bronze  (existe: True)
Ruta silver:  /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco/data/silver  (existe: True)
Ruta gold:    /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco/data/gold  (existe: True)
URL base API: https://api.open-me

In [2]:
import duckdb
import pandas as pd

## Funcion principal

In [3]:
def _con_duckdb():
    """Crea una conexion DuckDB en memoria para consultar los ficheros
    Parquet de Silver directamente por SQL (motor analitico in-process,
    segun exige el enunciado).
    """
    return duckdb.connect(database=":memory:")


def _patron_silver():
    """Patron glob que cubre todas las particiones de la capa Silver."""
    return str(RUTA_SILVER / "ciudad=*" / "anio=*" / "mes=*" / "datos.parquet")


def calcular_resumen_diario():
    """Resumen diario agregado por ubicacion: temperatura media, minima,
    maxima, lluvia total acumulada y velocidad media del viento.

    Devuelve: DataFrame con una fila por (ciudad, fecha).
    """
    patron = _patron_silver()
    con = _con_duckdb()
    consulta = f"""
        SELECT
            ciudad,
            CAST(fecha_hora_utc AS DATE) AS fecha,
            AVG(temperature_2m)  AS temperatura_media_c,
            MIN(temperature_2m)  AS temperatura_minima_c,
            MAX(temperature_2m)  AS temperatura_maxima_c,
            SUM(precipitation)   AS precipitacion_total_mm,
            AVG(wind_speed_10m)  AS viento_medio_kmh
        FROM read_parquet('{patron}', hive_partitioning=1)
        GROUP BY ciudad, CAST(fecha_hora_utc AS DATE)
        ORDER BY ciudad, fecha
    """
    return con.execute(consulta).fetchdf()


def calcular_media_movil_temperatura():
    """Metrica horaria continua de media movil a VENTANA_MEDIA_MOVIL_HORAS
    (24) horas de temperatura, por ciudad, usando una funcion de ventana
    de DuckDB.

    Devuelve: DataFrame con una fila por (ciudad, fecha_hora_utc).
    """
    patron = _patron_silver()
    con = _con_duckdb()
    consulta = f"""
        SELECT
            ciudad,
            fecha_hora_utc,
            temperature_2m,
            AVG(temperature_2m) OVER (
                PARTITION BY ciudad
                ORDER BY fecha_hora_utc
                ROWS BETWEEN {VENTANA_MEDIA_MOVIL_HORAS - 1} PRECEDING AND CURRENT ROW
            ) AS media_movil_24h_c
        FROM read_parquet('{patron}', hive_partitioning=1)
        ORDER BY ciudad, fecha_hora_utc
    """
    return con.execute(consulta).fetchdf()


def calcular_zscore_horario():
    """Z-Score horario de temperatura por ciudad frente a su historico
    cargado (media y desviacion tipica de toda la serie Silver disponible
    para esa ciudad). Permite detectar desviaciones termicas.

    Devuelve: DataFrame con una fila por (ciudad, fecha_hora_utc).
    """
    patron = _patron_silver()
    con = _con_duckdb()
    consulta = f"""
        WITH datos AS (
            SELECT ciudad, fecha_hora_utc, temperature_2m
            FROM read_parquet('{patron}', hive_partitioning=1)
        ),
        estadisticas AS (
            SELECT
                ciudad,
                AVG(temperature_2m) AS media_historica_c,
                STDDEV_POP(temperature_2m) AS desviacion_historica_c
            FROM datos
            GROUP BY ciudad
        )
        SELECT
            d.ciudad,
            d.fecha_hora_utc,
            d.temperature_2m,
            e.media_historica_c,
            e.desviacion_historica_c,
            CASE
                WHEN e.desviacion_historica_c = 0 OR e.desviacion_historica_c IS NULL THEN 0.0
                ELSE (d.temperature_2m - e.media_historica_c) / e.desviacion_historica_c
            END AS z_score_temperatura
        FROM datos d
        JOIN estadisticas e USING (ciudad)
        ORDER BY d.ciudad, d.fecha_hora_utc
    """
    return con.execute(consulta).fetchdf()


def escribir_gold(nombre_tabla, df):
    """Escribe una tabla Gold en Parquet, particionada por ciudad,
    sobrescribiendo por completo la tabla (recalculo idempotente desde
    Silver en cada ejecucion).
    """
    directorio = RUTA_GOLD / nombre_tabla
    directorio.mkdir(parents=True, exist_ok=True)
    for ciudad, grupo in df.groupby("ciudad"):
        sub = directorio / f"ciudad={ciudad}"
        sub.mkdir(parents=True, exist_ok=True)
        grupo.drop(columns=["ciudad"]).to_parquet(sub / "datos.parquet", index=False)
    return len(df)


def procesar_gold():
    """Orquesta el calculo y la escritura de las tres tablas Gold.

    Devuelve: diccionario con el numero de filas de cada tabla.
    """
    resumen_diario = calcular_resumen_diario()
    media_movil = calcular_media_movil_temperatura()
    zscore = calcular_zscore_horario()

    n_resumen = escribir_gold("resumen_diario", resumen_diario)
    n_media_movil = escribir_gold("media_movil_temp", media_movil)
    n_zscore = escribir_gold("zscore_horario", zscore)

    logger.info(
        f"Gold finalizado. resumen_diario: {n_resumen} filas | "
        f"media_movil_temp: {n_media_movil} filas | zscore_horario: {n_zscore} filas"
    )
    return {
        "resumen_diario": n_resumen,
        "media_movil_temp": n_media_movil,
        "zscore_horario": n_zscore,
    }

## Prueba rapida

Se calculan las tres tablas Gold sobre los datos reales ya presentes en Silver y se muestra una vista rapida de cada una, incluyendo una comprobacion de consistencia matematica manual del Z-Score (media 0 y desviacion 1 sobre la propia serie historica).

In [4]:
resumen_gold = procesar_gold()
print("Filas escritas por tabla Gold:", resumen_gold)

df_resumen_diario = calcular_resumen_diario()
print("\n--- resumen_diario (primeras filas) ---")
print(df_resumen_diario.head(5).to_string(index=False))

df_media_movil = calcular_media_movil_temperatura()
print("\n--- media_movil_temp (primeras filas de Madrid) ---")
print(df_media_movil[df_media_movil["ciudad"] == "Madrid"].head(5).to_string(index=False))

df_zscore = calcular_zscore_horario()
print("\n--- zscore_horario (primeras filas de Madrid) ---")
print(df_zscore[df_zscore["ciudad"] == "Madrid"].head(5).to_string(index=False))

# Comprobacion de consistencia matematica: para cada ciudad, la media y la
# desviacion tipica del z-score sobre su propia serie deben ser ~0 y ~1.
for ciudad, grupo in df_zscore.groupby("ciudad"):
    media_z = grupo["z_score_temperatura"].mean()
    std_z = grupo["z_score_temperatura"].std(ddof=0)
    assert abs(media_z) < 1e-6, f"Media del z-score de {ciudad} deberia ser ~0, es {media_z}"
    assert abs(std_z - 1.0) < 1e-6, f"Desviacion del z-score de {ciudad} deberia ser ~1, es {std_z}"
print("\nValidacion de consistencia matematica del Z-Score: OK (media~0, desviacion~1 por ciudad)")

Filas escritas por tabla Gold: {'resumen_diario': 75, 'media_movil_temp': 1680, 'zscore_horario': 1680}

--- resumen_diario (primeras filas) ---
   ciudad      fecha  temperatura_media_c  temperatura_minima_c  temperatura_maxima_c  precipitacion_total_mm  viento_medio_kmh
Barcelona 2026-09-22            22.478571                  16.1                  28.1                    4.05          9.421429
Barcelona 2026-09-23            22.091667                  16.1                  28.1                   17.63         10.058333
Barcelona 2026-09-24            22.141667                  16.2                  28.2                    5.54         10.687500
Barcelona 2026-09-25            22.191667                  16.2                  28.2                    0.00          9.462500
Barcelona 2026-09-26            22.241667                  16.3                  28.3                    0.17          9.679167

--- media_movil_temp (primeras filas de Madrid) ---
ciudad      fecha_hora_utc  temper

---
✅ **`gold_analytics.ipynb` verificado/a.**